# Colour-preference experiment — combined dataset preparation
Run `raw_data_preparation_color_preference.ipynb` first whenever a recording does not yet contain `XY_full.h5`. This notebook applies the spreadsheet exclusions, parses the extended trial labels and creates the combined pickle used by `visualising_color_choice.ipynb`.

### 0. Initialisation

#### 0.1 Locate the workflow and define data paths
The setup locates the repository automatically, uses `analysis_methods_color_preference.json`, points to the mounted LocustVR data, and defines where `locustvr_data.pkl` is saved. Set `save_pickle = False` only when the combined dataset should not be overwritten.

In [ ]:
from pathlib import Path

current_working_directory = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in [current_working_directory, *current_working_directory.parents]
     if (candidate / 'locustvr_extractor.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the UnityDataAnalysis repository root.')
workflow_dir = repo_root / 'analyses' / 'color_preference'
root_dir_data = "/Volumes/AG_Couzin-Fuchs/DATA/experiment_trackball_Optomotor/locustVR"
save_pickle = True
save_path = "/Users/jonny/Desktop/Uni/Master/collevtive behavior/VR_analysis/collor_choice"

#### 0.2 Import packages

In [ ]:
import os,json
import sys
from pathlib import Path

utilities_dir = repo_root.parent / 'utilities'
if not (utilities_dir / 'useful_tools.py').is_file():
    raise FileNotFoundError(f'Could not find useful_tools.py in {utilities_dir}')
for import_dir in (repo_root, utilities_dir):
    import_path = str(import_dir)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('default')
import importlib
from LocustVR_data_analysis import utils_local

importlib.reload(utils_local)
from LocustVR_data_analysis.utils_local import align_and_flip_heading, reindex_t_by_state_transition, align_trajectories, \
    compute_directness_and_direction, flip_symmetric_states

from useful_tools import select_animals_gpt

json_file = workflow_dir / 'analysis_methods_color_preference.json'

with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())

exp_name = analysis_methods.get("experiment_name")
variable_name = 'condition_code'

TRIAL_LABEL_PATTERN = (
    r'^(?:T\d+_)?CD(?P<distance_cm>\d+)_CS(?P<condition_code>\d+)_'
    r'S(?P<label_state>\d+)_L(?P<left_texture>[^_]+)_'
    r'R(?P<right_texture>[^_]+)_BT(?P<bait_texture>[^_]+)$'
)
LABEL_COLUMNS = [
    'distance_cm', 'condition_code', 'label_state',
    'left_texture', 'right_texture', 'bait_texture'
]

def add_trial_label_columns(data):
    """Parse the extended LocustVR label while keeping trial_label unchanged."""
    if 'trial_label' not in data.columns:
        raise KeyError("The input file does not contain a 'trial_label' column.")

    result = data.copy()
    extracted = result['trial_label'].astype(str).str.extract(TRIAL_LABEL_PATTERN)
    invalid_labels = result.loc[extracted.isna().all(axis=1), 'trial_label'].dropna().unique()
    if len(invalid_labels) > 0:
        examples = ', '.join(map(str, invalid_labels[:5]))
        raise ValueError(f'Unrecognised trial label(s), for example: {examples}')

    for column in ['distance_cm', 'condition_code', 'label_state']:
        extracted[column] = pd.to_numeric(extracted[column], errors='coerce').astype('Int64')

    for column in LABEL_COLUMNS:
        if column in result.columns:
            result[column] = result[column].fillna(extracted[column])
        else:
            result[column] = extracted[column]

    return result

### 1. Select and prepare colour-preference data

#### 1.1 Load metadata and select sessions
The LocustVR Google Sheet supplies animal metadata and the optional inclusive cutoff in `Exclude trials after trial X:`. Empty cutoff cells keep every available trial. The current `dir_lis_2026 = dir_list[-19:]` line then restricts processing to the final 19 discovered session folders; review it whenever recordings are added.

In [ ]:
# Define the data source and discover HDF5 directories.
dir_list = []
file_type = '.h5'
using_google_sheet = True
sheet_name = 'LocustVR'
TRIAL_CUTOFF_COLUMN = 'Exclude trials after trial X:'

if using_google_sheet:
    database_id = '1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8'
    url = (
        f'https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq'
        f'?tqx=out:csv&sheet={sheet_name}'
    )
    sheet_df = pd.read_csv(url)
else:
    excel_file_path = (
        '/Volumes/AG_Couzin-Fuchs/DATA/experiment_trackball_Optomotor/'
        'Locusts Management.xlsx'
    )
    print(f'Using local database {excel_file_path}; it might be outdated.')
    sheet_df = pd.read_excel(excel_file_path, sheet_name=sheet_name)

if TRIAL_CUTOFF_COLUMN not in sheet_df.columns:
    raise KeyError(f'Missing spreadsheet column: {TRIAL_CUTOFF_COLUMN}')

cutoff_table = sheet_df[['ID', TRIAL_CUTOFF_COLUMN]].copy()
cutoff_table['sheet_id'] = cutoff_table['ID'].astype('string').str.strip()
raw_cutoff = cutoff_table[TRIAL_CUTOFF_COLUMN].replace(
    r'^\s*$', pd.NA, regex=True
)
cutoff_table['trial_cutoff'] = pd.to_numeric(raw_cutoff, errors='coerce')
invalid_text = raw_cutoff.notna() & cutoff_table['trial_cutoff'].isna()
invalid_number = (
    cutoff_table['trial_cutoff'].notna()
    & (
        cutoff_table['trial_cutoff'].lt(0)
        | cutoff_table['trial_cutoff'].mod(1).ne(0)
    )
)
if invalid_text.any() or invalid_number.any():
    bad_rows = cutoff_table.loc[
        invalid_text | invalid_number, ['ID', TRIAL_CUTOFF_COLUMN]
    ]
    raise ValueError(
        'Trial cutoffs must be empty or non-negative whole numbers:\n'
        + bad_rows.to_string(index=False)
    )

conflicts = (
    cutoff_table.dropna(subset=['sheet_id', 'trial_cutoff'])
    .groupby('sheet_id')['trial_cutoff'].nunique()
)
if conflicts.gt(1).any():
    conflict_ids = conflicts.index[conflicts.gt(1)].tolist()
    raise ValueError(f'Conflicting trial cutoffs for IDs: {conflict_ids}')

trial_cutoff_by_id = (
    cutoff_table.dropna(subset=['sheet_id', 'trial_cutoff'])
    .groupby('sheet_id')['trial_cutoff'].first().astype(int).to_dict()
)
all_sheet_ids = (
    sheet_df['ID'].dropna().astype(str).str.strip().unique()
)

if analysis_methods.get('load_individual_data') == True:
    if analysis_methods.get('select_animals_by_condition') == True:
        animal_of_interest = select_animals_gpt(
            sheet_df,
            'Excluding this animal from analysis (Usually when animals die or molt, T/F)',
            'F',
        )
    else:
        animal_of_interest = sheet_df
    selected_ids = (
        animal_of_interest['ID'].dropna().astype(str).str.strip().unique()
    )
    print(f'Valid animal IDs:\n{selected_ids}')
    dir_list = [
        root.replace('\\', '/')
        for root, _, files in os.walk(root_dir_data)
        if any(ID in Path(root).parts for ID in selected_ids)
        and any(file.endswith(file_type) for file in files)
    ]
else:
    animal_of_interest = sheet_df
    for root, dirs, files in os.walk(root_dir_data):
        for folder in dirs:
            folder_path = os.path.join(root, folder)
            if any(
                name.endswith(file_type) for name in os.listdir(folder_path)
            ):
                dir_list.append(folder_path.replace('\\', '/'))

dir_list.sort()
dir_to_sheet_id = {}
for this_dir in dir_list:
    matches = [ID for ID in all_sheet_ids if ID in Path(this_dir).parts]
    if len(matches) > 1:
        raise ValueError(
            f'Multiple spreadsheet IDs match {this_dir}: {matches}'
        )
    if len(matches) == 1:
        dir_to_sheet_id[this_dir] = matches[0]

print('Directories:')
for this_dir in dir_list:
    sheet_id = dir_to_sheet_id.get(this_dir, 'not in spreadsheet')
    cutoff = trial_cutoff_by_id.get(sheet_id)
    suffix = f' — keep through trial {cutoff}' if cutoff is not None else ''
    print(f'{this_dir} [{sheet_id}]{suffix}')


In [ ]:
dir_lis_2026 = dir_list[-19:] 
#dir_list = dir_list[-6:] 
dir_lis_2026


#### 1.2 Build the combined colour-preference dataset
Input is `XY_full.h5`. Trials after an animal's spreadsheet cutoff are removed first. The complete `trial_label` is retained and additionally split into `distance_cm`, `condition_code`, `label_state`, `left_texture`, `right_texture`, and `bait_texture`; trajectories and headings are then aligned before concatenation.


In [ ]:
file_name = "/XY_full.h5"
pd_list = []
for index, this_dir in enumerate(dir_lis_2026):
    print(index, this_dir)
    this_pd = pd.read_hdf(this_dir + file_name)
    this_pd['animal_id'] = index
    this_pd = reindex_t_by_state_transition(this_pd, transition_from=0, transition_to=(1, 2))

    sheet_id = dir_to_sheet_id.get(this_dir)
    trial_cutoff = trial_cutoff_by_id.get(sheet_id) if sheet_id else None
    if trial_cutoff is not None:
        trial_ids = pd.to_numeric(this_pd['trial_id'], errors='coerce')
        invalid_trial_ids = this_pd['trial_id'].notna() & trial_ids.isna()
        if invalid_trial_ids.any():
            examples = this_pd.loc[invalid_trial_ids, 'trial_id'].unique()[:5]
            raise ValueError(
                f'Non-numeric trial_id values for {sheet_id}: {examples}'
            )
        excluded_trials = sorted(
            trial_ids.loc[trial_ids.gt(trial_cutoff)].dropna().astype(int).unique()
        )
        this_pd = this_pd.loc[trial_ids.le(trial_cutoff)].copy()
        print(
            f'  {sheet_id}: keeping trials through {trial_cutoff}; '
            f'excluded {len(excluded_trials)} later trial(s)'
        )
    this_pd = align_and_flip_heading(this_pd)
    this_pd = add_trial_label_columns(this_pd)
    this_pd = align_trajectories(this_pd)
    this_pd = flip_symmetric_states(this_pd)
    this_pd = compute_directness_and_direction(this_pd)
    pd_list.append(this_pd)

if not pd_list:
    raise ValueError('No XY_full.h5 files were loaded from the selected directories.')

df = pd.concat(pd_list, ignore_index=True)

label_preview = (
    df[['trial_label'] + LABEL_COLUMNS]
    .drop_duplicates()
    .sort_values(['condition_code', 'label_state', 'left_texture', 'right_texture'])
)
display(label_preview.head(20))

if save_pickle:
    os.makedirs(save_path, exist_ok=True)
    df.to_pickle(os.path.join(save_path, 'locustvr_data.pkl'))

### 2. Inspect an existing combined dataset
These cells reload the saved pickle and summarize the number of trials per animal.

In [ ]:
df = pd.read_pickle(os.path.join(save_path, 'locustvr_data.pkl'))


In [ ]:
df.head(20)


In [ ]:
grouped = df.groupby("animal_id")["trial_id"].agg(["max", "nunique"])
print(grouped)